# 03c · Línea base con un LLM local por *prompting* (corrección del PSet #3)

Observación del PSet #3: faltaba ejecutar un LLM por *prompting*, y la falta de presupuesto para la API no descartaba un **modelo local**. Aquí se usa **Qwen2.5-1.5B-Instruct** servido con **Ollama** en Docker (sin costo, sin enviar datos fuera de la máquina), en modo *zero-shot*: no se entrena con nuestros datos.

- **Pregunta al modelo:** probabilidad (0-100) de que la reseña tenga 1 o 2 estrellas, es decir, el mismo target `es_negativa`.
- **Evaluación:** la misma muestra fija del hold-out temporal que usan las pruebas de CI (`tests/fixtures/muestra.csv`: 200 negativas + 200 no negativas, semilla 42), comparada con el champion del PSet #3 y con la regla de negocio.
- **Umbral:** se elige con 300 reseñas del entrenamiento (nunca con la prueba), igual que en el PSet #3.

Requisitos: `docker compose --profile llm up -d ollama` y `docker compose exec ollama ollama pull qwen2.5:1.5b`.

In [1]:
import os, re, time, json, requests, numpy as np, pandas as pd, mlflow, mlflow.sklearn
from sklearn.metrics import average_precision_score, roc_auc_score
from src.datos import cargar_obt_entrenamiento
from src.pipeline import *
from src.experimento import configurar, limpiar_fase, commit_git, NOMBRE_EXPERIMENTO, NOMBRE_MODELO
OLLAMA = os.environ.get("OLLAMA_URL", "http://ollama:11434"); MODELO_LLM = "qwen2.5:1.5b"
df = agregar_features(cargar_obt_entrenamiento())
train, test = split_temporal(df, 0.2)
evalua = pd.concat([test[test[TARGET] == k].sample(200, random_state=SEED) for k in (0, 1)]).sort_values("review_id")
calib = train.tail(int(len(train) * 0.25)).sample(300, random_state=SEED)
print("Evaluación:", len(evalua), "reseñas del hold-out | calibración del umbral:", len(calib), "reseñas del entrenamiento")
print("Ollama:", requests.get(f"{OLLAMA}/api/tags", timeout=10).json())

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/usr/local/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


/usr/local/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Evaluación: 400 reseñas del hold-out | calibración del umbral: 300 reseñas del entrenamiento
Ollama: {'models': [{'name': 'qwen2.5:1.5b', 'model': 'qwen2.5:1.5b', 'modified_at': '2026-11-24T18:53:00.455171682Z', 'size': 986061892, 'digest': '65ec06548149b04c096a120e4a6da9d4017ea809c91734ea5631e89f96ddc57b', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '1.5B', 'quantization_level': 'Q4_K_M', 'context_length': 32768, 'embedding_length': 1536, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools']}]}


In [2]:
PROMPT = """Eres analista de experiencia de clientes de un banco en Ecuador.
Lee la reseña que un cliente publicó sobre la app del banco en Google Play y estima la probabilidad,
de 0 a 100, de que el cliente la haya calificado con 1 o 2 estrellas (una queja que requiere atención).
Responde solo con un número entero entre 0 y 100, sin texto adicional.

Banco: {banco}
Reseña: "{texto}"
Probabilidad:"""

def puntuar_llm(fila):
    t0 = time.perf_counter()
    r = requests.post(f"{OLLAMA}/api/generate", timeout=120, json={
        "model": MODELO_LLM, "prompt": PROMPT.format(banco=fila["banco"], texto=fila[TEXTO][:1500]),
        "stream": False, "options": {"temperature": 0, "num_predict": 5, "seed": SEED}})
    ms = (time.perf_counter() - t0) * 1000
    m = re.search(r"\d{1,3}", r.json().get("response", ""))
    return (min(int(m.group()), 100) / 100 if m else np.nan), ms

def puntuar_todo(d, etiqueta):
    res = []
    for i, fila in enumerate(d.to_dict("records"), 1):
        res.append(puntuar_llm(fila))
        if i % 100 == 0: print(f"{etiqueta}: {i}/{len(d)}")
    p, ms = zip(*res)
    return np.array(p, dtype=float), np.array(ms)

p_cal, _ = puntuar_todo(calib, "calibración")
p_eva, ms_eva = puntuar_todo(evalua, "evaluación")
print(f"Respuestas sin número: calibración {np.isnan(p_cal).mean():.1%} | evaluación {np.isnan(p_eva).mean():.1%}")

calibración: 100/300


calibración: 200/300


calibración: 300/300


evaluación: 100/400


evaluación: 200/400


evaluación: 300/400


evaluación: 400/400
Respuestas sin número: calibración 0.0% | evaluación 0.0%


## Comparación en la misma muestra
Si el LLM no devuelve un número se usa la prevalencia de calibración (no se descarta la fila, para comparar sobre las mismas 400 reseñas).

In [3]:
prev = calib[TARGET].mean()
p_cal_ok, p_eva_ok = np.nan_to_num(p_cal, nan=prev), np.nan_to_num(p_eva, nan=prev)
umbral_llm, _ = mejor_umbral(calib[TARGET].values, p_cal_ok)
y = evalua[TARGET].values

cli = mlflow.MlflowClient(); configurar()
exp = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
fin = mlflow.search_runs([exp.experiment_id], filter_string="tags.fase = 'final' and tags.rol = 'champion'").iloc[0]
champion = mlflow.sklearn.load_model(f"runs:/{fin['run_id']}/modelo"); umbral_ch = fin["metrics.umbral_aplicado"]
t0 = time.perf_counter(); p_ch = champion.predict_proba(evalua[FEATURES])[:, 1]; ms_ch = (time.perf_counter() - t0) / len(evalua) * 1000

def fila(nombre, p, u, ms):
    pred = (p >= u).astype(int) if p is not None else (evalua["n_neg_lexico"] >= 1).astype(int).values
    return {"modelo": nombre, "PR-AUC": average_precision_score(y, p) if p is not None else np.nan,
            "ROC-AUC": roc_auc_score(y, p) if p is not None else np.nan, "umbral": u,
            "costo por reseña": costo_negocio(y, pred), "recall": ((pred == 1) & (y == 1)).sum() / y.sum(),
            "precisión": ((pred == 1) & (y == 1)).sum() / max(pred.sum(), 1), "ms por reseña": ms}
tabla = pd.DataFrame([fila(f"LLM local ({MODELO_LLM}, zero-shot)", p_eva_ok, umbral_llm, float(np.mean(ms_eva))),
                      fila("Champion PSet #3 (embeddings + LogReg)", p_ch, umbral_ch, ms_ch),
                      fila("Regla de negocio (léxico)", None, None, 0.0)]).set_index("modelo")
tabla.round(4)

,PR-AUC,ROC-AUC,umbral,costo por reseña,recall,precisión,ms por reseña
modelo,,,,,,,
"LLM local (qwen2.5:1.5b, zero-shot)",0.5582,0.5654,0.11,0.545,0.945,0.5369,236.6846
Champion PSet #3 (embeddings + LogReg),0.9346,0.9524,0.22,0.160,0.965,0.8694,1.9896
Regla de negocio (léxico),NaN,NaN,NaN,0.820,0.685,0.9133,0.0000


In [4]:
limpiar_fase("candidato", familias=["llm_local"])
with mlflow.start_run(run_name=f"candidato-LLM-local-{MODELO_LLM}"):
    mlflow.set_tags({"fase": "candidato", "familia": "llm_local", "commit_git": commit_git(),
                     "nota": "zero-shot por prompting; evaluado en la muestra fija del hold-out (no en CV)"})
    mlflow.log_params({"modelo_llm": MODELO_LLM, "servidor": "ollama", "temperature": 0, "n_eval": len(evalua),
                       "n_calibracion": len(calib), "semilla": SEED})
    mlflow.log_text(PROMPT, "prompt.txt")
    r = tabla.loc[f"LLM local ({MODELO_LLM}, zero-shot)"]
    mlflow.log_metrics({"muestra_pr_auc": r["PR-AUC"], "muestra_roc_auc": r["ROC-AUC"], "umbral_costo": umbral_llm,
                        "muestra_costo": r["costo por reseña"], "muestra_recall": r["recall"],
                        "ms_inferencia_por_resena": r["ms por reseña"], "pct_respuestas_invalidas": float(np.isnan(p_eva).mean())})
mejor = tabla["costo por reseña"].idxmin()
print(f"Menor costo en la muestra: {mejor}.")
print(f"El LLM tarda {tabla.iloc[0]['ms por reseña']:.0f} ms por reseña frente a {tabla.iloc[1]['ms por reseña']:.1f} ms del champion.")

🏃 View run candidato-LLM-local-qwen2.5:1.5b at: http://mlflow:5000/#/experiments/2/runs/e9e4edf56ff04dfca4de527cd02437b6
🧪 View experiment at: http://mlflow:5000/#/experiments/2
Menor costo en la muestra: Champion PSet #3 (embeddings + LogReg).
El LLM tarda 237 ms por reseña frente a 2.0 ms del champion.


**Lectura.** El LLM local no se ajusta con nuestras reseñas: sirve como línea base de "conocimiento general" sin entrenamiento. La decisión de mantener el champion se sostiene si su costo en la misma muestra es menor y su latencia compatible con el patrón batch; ambos valores están en la tabla anterior y en MLflow (`fase = candidato`, `familia = llm_local`).